# TEPS x OECD Social-Emotional Skills: Scoring Workflow (Waves 1 and 2)

Raw TEPS data are **not** included. Set the paths in the configuration cell below to your own copies of the files.

In [ ]:
import os
import warnings
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FactorAnalysis
from sklearn.impute import SimpleImputer

import PCA_detail as PCA            # PCA module
import EFA as EFA                   # EFA module
from teps_labels_util import TEPSLabelsManager  # variable-label helper

warnings.filterwarnings('ignore')
lib = importlib
importlib.reload(PCA)


def handle_special_missing_values(df):
    # Identify numeric columns
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns

    for col in numeric_columns:
        # Replace 97 and 99 with NaN in the column values
        df[col] = df[col].replace([97, 99], np.nan)

    return df


def new_handle_special_missing_values(df):
    # Identify numeric columns
    numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns

    for col in numeric_columns:
        # Replace 6, 97 and 99 with NaN in the column values
        df[col] = df[col].replace([6, 97, 99], np.nan)

    return df

## 1. Setup and data loading

In [ ]:
# ---- Configuration: edit these paths for your own machine ----
DATA_DIR = Path("path/to/TEPS_raw_data")      # folder holding the raw TEPS .dta files
OUT_DIR  = Path("path/to/output")             # folder for derived files (do not commit these)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# File names of the raw TEPS Stata files (adjust to your local copies)
W1_STUDENT_FILE  = "w1_student.dta"           # Wave 1, junior high student questionnaire
W1_EVAL_FILE     = "w1_student_evaluation.dta"  # Wave 1, student performance assessment
W1_PARENT_FILE   = "w1_parent.dta"            # Wave 1, parent questionnaire
W2_STUDENT_FILE  = "w2_student.dta"           # Wave 2, junior high student questionnaire
W2_EVAL_FILE     = "w2_student_evaluation.dta"  # Wave 2, student performance assessment
W2_PARENT_FILE   = "w2_parent.dta"            # Wave 2, parent questionnaire

In [ ]:
## Wave 1
student, meta = pyreadstat.read_dta(DATA_DIR / W1_STUDENT_FILE)
evaluate, meta = pyreadstat.read_dta(DATA_DIR / W1_EVAL_FILE)
parents, meta = pyreadstat.read_dta(DATA_DIR / W1_PARENT_FILE)

## Wave 2
student2, meta = pyreadstat.read_dta(DATA_DIR / W2_STUDENT_FILE)
student2 = handle_special_missing_values(student2)
evaluate2, meta = pyreadstat.read_dta(DATA_DIR / W2_EVAL_FILE)
evaluate2 = new_handle_special_missing_values(evaluate2)
parents2, meta = pyreadstat.read_dta(DATA_DIR / W2_PARENT_FILE)
parents2 = handle_special_missing_values(parents2)

student = handle_special_missing_values(student)
parents = handle_special_missing_values(parents)

## 2. OECD skill crosswalk (Wave 1)

The crosswalk includes self-report items on a 4-point scale (the "since this semester, have you..." / "does this describe you" item sets, `w1s508`-`w1s555`). Each item is handled in one of two ways:

- **FLIP**: the original coding runs from `1 = never / fits very well (positively worded)` to `4 = often / does not fit at all`, so it is reversed with `5 - raw value`. After reversal, a higher score means a higher skill level.
- **NOFLIP**: the item is itself negatively worded (e.g. "gives me a headache"), so the original coding already means "higher score = higher skill" and no reversal is needed.

The variable classification follows the TEPS non-cognitive measure document (kept locally, not included in this repository).

In [ ]:
# 1. Category
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
openness_vars = [ 'w1s538', 'w1s419', 'w1s417', 'w1s418', 'w1s552']
openness1 = student[['stud_id']+openness_vars].copy()
task_vars = ['w1s547', 'w1s508', 'w1s514','w1s509','w1s535','w1s512','w1s536','w1s537','w1s545']
task1 = student[['stud_id']+task_vars].copy()
# reverse specific variables
task1['w1s545'] = 5 - task1['w1s545']
task1['w1s512'] = 5 - task1['w1s512']

engage_vars=['w1s411', 'w1s402']
engage1 = student[['stud_id']+engage_vars].copy()
engage1['w1s402']=3-engage1['w1s402']

colla_vars = ['w1s546', 'w1s548','w1s510','w1s420', 'w1s539','w1s516', 'w1s544']
colla1 = student[['stud_id']+colla_vars].copy()
colla1['w1s510'] = 5 - colla1['w1s510']

emotion_vars = ['w1s540','w1s515','w1s518','w1s521','w1s526','w1s527','w1s517','w1s522','w1s523','w1s555','w1s519', 'w1s510']
emotion1 = student[['stud_id'] + emotion_vars].copy()
neg_vars = ['w1s515','w1s518','w1s521','w1s526','w1s527','w1s517','w1s522','w1s523','w1s519', 'w1s510']
emotion1[neg_vars] = 5 - emotion1[neg_vars]

compound_vars = ['w1s544', 'w1s546','w1s547' ,'w1s548', 'w1s536', 'w1s537','w1s550']
compound1 = student[['stud_id']+compound_vars].copy()

### Internal consistency (Cronbach's alpha), Wave 1

In [ ]:
## Test Cronbach alpha
import pandas as pd
import Cronbach_Alpha as ca
lib.reload(ca)

scale_dict = {
    'Openness': openness1,
    'Task Performance': task1,
    'Engage with Others': engage1,
    'Collaboration': colla1,
    'Emotional Regulation': emotion1,
    'Compound Skills': compound1,
}

cronbach_results = []

for scale_name, df in scale_dict.items():
    df_new = df.drop('stud_id', axis=1)
    df_clean = df_new.dropna()
    n_items = df_new.shape[1]
    alpha = ca.cronbach_alpha(df_clean)
    cronbach_results.append({
        'Scale': scale_name,
        'Cronbach_alpha': alpha,
        'N_items': n_items,
    })

cronbach_table = pd.DataFrame(cronbach_results)
cronbach_table['Cronbach_alpha'] = cronbach_table['Cronbach_alpha'].round(3)
cronbach_table

### PCA by domain, Wave 1 (exploratory)

In [ ]:
import importlib
import PCA_detail as PCA
importlib.reload(PCA)
# Openness
openness1_clean = openness1.drop(columns=["stud_id"], errors="ignore")
openness1_result = PCA.perform_pca_analysis(openness1_clean, 0.4)

In [ ]:
# Task Performance
task1_clean = task1.drop(columns=["stud_id"], errors="ignore")
task1_result = PCA.perform_pca_analysis(task1_clean, 0.2)

In [ ]:
# Collaboration
colla1_clean = colla1.drop(columns=["stud_id"], errors="ignore")
colla1_result = PCA.perform_pca_analysis(colla1_clean, 0.3)

In [ ]:
# Emotional Regulation
emotion1_clean = emotion1.drop(columns=["stud_id"], errors="ignore")
emotion1_result = PCA.perform_pca_analysis(emotion1_clean, 0.3)

In [ ]:
engage1_clean = engage1.drop(columns=["stud_id"], errors="ignore")
engage1_result = PCA.perform_pca_analysis(engage1_clean, 0.3)

In [ ]:
# Compound Skills
compound1_clean = compound1.drop(columns=["stud_id"], errors="ignore")
compound1_result = PCA.perform_pca_analysis(compound1_clean, 0.3)

### PCA scores by domain, Wave 1 (exported for analysis)

In [ ]:
# Openness
openness1_clean = openness1.drop(columns=["stud_id"], errors="ignore")
openness1_result = PCA.perform_pca_analysis(openness1_clean, 0.4)
openness1_result["merged_table"].to_csv(
    str(OUT_DIR / "Openness_pca.csv"),
    encoding="utf-8-sig"
)
## Make sure stud_id is not missing
selected_items = openness1_result["merged_table"].index.tolist()
valid_index = openness1_clean[selected_items].dropna().index

openness1_score = openness1_result["pca_score"].copy()
openness1_score["stud_id"] = openness1.loc[valid_index, "stud_id"].to_numpy()
openness1_score = openness1_score[["stud_id", "PC1 Score"]]

openness1_score.to_stata(
    str(OUT_DIR / "Openness_PCA1.dta"),
    write_index=False
)


# Task Performance
task1_clean = task1.drop(columns=["stud_id"], errors="ignore")
task1_result = PCA.perform_pca_analysis(task1_clean, 0.1)

task1_result["merged_table"].to_csv(
    str(OUT_DIR / "task1_pca.csv"),
    encoding="utf-8-sig"
)
## Make sure stud_id is not missing
selected_items = task1_result["merged_table"].index.tolist()
valid_index = task1_clean[selected_items].dropna().index

task1_score = task1_result["pca_score"].copy()
task1_score["stud_id"] = task1.loc[valid_index, "stud_id"].to_numpy()
task1_score = task1_score[["stud_id", "PC1 Score"]]

task1_score.to_stata(
    str(OUT_DIR / "Task_PCA.dta"),
    write_index=False
)


# Collaboration
colla1_clean = colla1.drop(columns=["stud_id"], errors="ignore")
colla1_result = PCA.perform_pca_analysis(colla1_clean, 0.3)

colla1_result["merged_table"].to_csv(
    str(OUT_DIR / "colla1_pca.csv"),
    encoding="utf-8-sig"
)
## Make sure stud_id is not missing
selected_items = colla1_result["merged_table"].index.tolist()
valid_index = colla1_clean[selected_items].dropna().index

colla1_score = colla1_result["pca_score"].copy()
colla1_score["stud_id"] = colla1.loc[valid_index, "stud_id"].to_numpy()
colla1_score = colla1_score[["stud_id", "PC1 Score"]]

colla1_score.to_stata(
    str(OUT_DIR / "Colla1_PCA.dta"),
    write_index=False
)


# Emotional Regulation
emotion1_clean = emotion1.drop(columns=["stud_id"], errors="ignore")
emotion1_result = PCA.perform_pca_analysis(emotion1_clean, 0.3)

emotion1_result["merged_table"].to_csv(
    str(OUT_DIR / "emotion1_pca.csv"),
    encoding="utf-8-sig"
)
## Make sure stud_id is not missing
selected_items = emotion1_result["merged_table"].index.tolist()
valid_index = emotion1_clean[selected_items].dropna().index

emotion1_score = emotion1_result["pca_score"].copy()
emotion1_score["stud_id"] = emotion1.loc[valid_index, "stud_id"].to_numpy()
emotion1_score = emotion1_score[["stud_id", "PC1 Score"]]

emotion1_score.to_stata(
    str(OUT_DIR / "Emotion1_PCA.dta"),
    write_index=False
)

# Compound Skills
compound1_clean = compound1.drop(columns=["stud_id"], errors="ignore")
compound1_result = PCA.perform_pca_analysis(compound1_clean, 0.3)

compound1_result["merged_table"].to_csv(
    str(OUT_DIR / "compound1_pca.csv"),
    encoding="utf-8-sig"
)
## Make sure stud_id is not missing
selected_items = compound1_result["merged_table"].index.tolist()
valid_index = compound1_clean[selected_items].dropna().index

compound1_score = compound1_result["pca_score"].copy()
compound1_score["stud_id"] = compound1.loc[valid_index, "stud_id"].to_numpy()
compound1_score = compound1_score[["stud_id", "PC1 Score"]]

compound1_score.to_stata(
    str(OUT_DIR / "Compound1_PCA.dta"),
    write_index=False
)

#### Save the cleaned item data for R (IRT scores)

In [ ]:
out_dir = OUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)
openness1.to_csv(out_dir / "Openness.csv", index=False, encoding="utf-8-sig")
task1.to_csv(out_dir / "TaskPerformance.csv", index=False, encoding="utf-8-sig")
engage1.to_csv(out_dir / "EngageWithOthers.csv", index=False, encoding="utf-8-sig")
colla1.to_csv(out_dir / "Collaboration.csv", index=False, encoding="utf-8-sig")
emotion1.to_csv(out_dir / "EmotionalRegulation.csv", index=False, encoding="utf-8-sig")
compound1.to_csv(out_dir / "CompoundSkills.csv", index=False, encoding="utf-8-sig")

## 2b. OECD skill crosswalk (Wave 2)

Same logic as Wave 1, applied to the Wave 2 student questionnaire (`w2s...` items). Items on a 4-point scale are reversed with `5 - raw value` where the original coding runs in the opposite direction of the skill. Items with other response formats are recoded explicitly below.

The variable classification follows the TEPS non-cognitive measure document (kept locally, not included in this repository).

In [ ]:
# Wave 2
# 1. Category
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()# Items that truly belong to Openness

engage_vars2 = ["w2s426", "w2s407", "w2s419", "w2s126"]  # Engage with Others
# w2s432 (not enough sleep) should be merged into the Emotional Regulation group

engage2 = student2[["stud_id"] + engage_vars2].copy()

engage2["w2s426"] = 5 - engage2["w2s426"]   # frequency item, reverse
engage2["w2s419"] = 5 - engage2["w2s419"]   # agreement item, reverse
engage2["w2s407_assertive"] = (engage2["w2s407"] == 3).astype(float)
engage2=engage2.drop('w2s407', axis=1)

In [ ]:
task_vars = ["w2s442", "w2s422","w2s413", "w2s433","w2s124","w2s125","w2s130","w2s441","w2s439","w2s440"]
task2 = student2[["stud_id"] + task_vars].copy()
# reverse specific variables
task2["w2s433"] = 5 - task2["w2s433"]
task2["w2s125"] = 5 - task2["w2s125"]
task2["w2s130"] = 5 - task2["w2s130"]
task2["w2s422"] = 5 - task2["w2s422"]
# recode 2 <-> 3
task2["w2s413"] = task2["w2s413"].replace({2: 3, 3: 2})
for col in ["w2s439", "w2s440", "w2s441", "w2s442"]:
    task2[col] = task2[col].replace({1: 4, 4: 1})

In [ ]:
colla_vars2 = ["w2s408", "w2s409","w2s412","w2s421", "w2s423", "w2s123", "w2s127", 'w2s426']
colla2 = student2[["stud_id"] + colla_vars2].copy()
# reverse 1-4 scale items
for col in ["w2s421", "w2s423", "w2s123", "w2s127", 'w2s426']:
    colla2[col] = 5 - colla2[col]

# w2s408
map_408 = {
    3: 3,
    1: 2,
    5: 1,
    2: 0,
    4: 0,
}

colla2["w2s408"] = colla2["w2s408"].map(map_408)

# w2s409
map_409 = {
    2: 2,
    3: 1,
    1: 0,
}

colla2["w2s409"] = colla2["w2s409"].map(map_409)

# w2s412
map_412 = {
    1: 3,
    2: 2,
    3: 1,
    4: 0,
}

colla2["w2s412"] = colla2["w2s412"].map(map_412)
emotion_vars2 = ["w2s415","w2s429","w2s431","w2s434","w2s427","w2s430","w2s436","w2s458","w2s428", "w2s437","w2s438", "w2s127"]
emotion2 = student2[["stud_id"] + emotion_vars2].copy()
# reverse all emotion items, assuming they are 1-4 scale
emotion2[emotion_vars2] = 5 - emotion2[emotion_vars2]

compound_vars2 = ["w2s418", "w2s424", "w2s416", "w2s417"]
compound2 = student2[["stud_id"] + compound_vars2].copy()
# positively worded items: reverse
for v in ["w2s418", "w2s424", "w2s416"]:
    compound2[v] = 5 - compound2[v]

### Internal consistency (Cronbach's alpha), Wave 2

In [ ]:
import pandas as pd
import Cronbach_Alpha as ca
lib.reload(ca)

scale_dict = {
    'Task Performance': task2,
    'Collaboration': colla2,
    'Emotional Regulation': emotion2,
    'Compound Skills': compound2,
    "Engagement": engage2,
}

cronbach_results2 = []

for scale_name, df in scale_dict.items():
    df_new = df.drop('stud_id', axis=1)
    df_clean = df_new.dropna()
    n_items = df_new.shape[1]
    alpha = ca.cronbach_alpha(df_clean)
    cronbach_results2.append({
        'Scale': scale_name,
        'Cronbach_alpha': alpha,
        'N_items': n_items,
    })

cronbach_table2 = pd.DataFrame(cronbach_results2)
cronbach_table2['Cronbach_alpha'] = cronbach_table2['Cronbach_alpha'].round(3)
cronbach_table2

In [ ]:
out_dir = OUT_DIR
out_dir.mkdir(parents=True, exist_ok=True)
task2.to_csv(out_dir / "TaskPerformance2.csv", index=False, encoding="utf-8-sig")
engage2.to_csv(out_dir / "EngageWithOthers2.csv", index=False, encoding="utf-8-sig")
colla2.to_csv(out_dir / "Collaboration2.csv", index=False, encoding="utf-8-sig")
emotion2.to_csv(out_dir / "EmotionalRegulation2.csv", index=False, encoding="utf-8-sig")
compound2.to_csv(out_dir / "CompoundSkills2.csv", index=False, encoding="utf-8-sig")

### PCA scores by domain, Wave 2

In [ ]:
import importlib
import PCA_detail as PCA
importlib.reload(PCA)
# Task Performance Wave 2
task2_clean = task2.drop(columns=["stud_id"], errors="ignore")
task2_result = PCA.perform_pca_analysis(task2_clean, 0.21)

task2_result["merged_table"].to_csv(
    str(OUT_DIR / "task2_pca.csv"),
    encoding="utf-8-sig"
)

selected_items = task2_result["merged_table"].index.tolist()
valid_index = task2_clean[selected_items].dropna().index

task2_score = task2_result["pca_score"].copy()
task2_score["stud_id"] = task2.loc[valid_index, "stud_id"].to_numpy()
task2_score = task2_score[["stud_id", "PC1 Score"]]

task2_score.to_stata(
    str(OUT_DIR / "Task2_PCA.dta"),
    write_index=False
)
# Collaboration Wave 2
colla2_clean = colla2.drop(columns=["stud_id"], errors="ignore")
colla2_result = PCA.perform_pca_analysis(colla2_clean, 0.21)

colla2_result["merged_table"].to_csv(
    str(OUT_DIR / "colla2_pca.csv"),
    encoding="utf-8-sig"
)

selected_items = colla2_result["merged_table"].index.tolist()
valid_index = colla2_clean[selected_items].dropna().index

colla2_score = colla2_result["pca_score"].copy()
colla2_score["stud_id"] = colla2.loc[valid_index, "stud_id"].to_numpy()
colla2_score = colla2_score[["stud_id", "PC1 Score"]]

colla2_score.to_stata(
    str(OUT_DIR / "Colla2_PCA.dta"),
    write_index=False
)
# Emotional Regulation Wave 2
emotion2_clean = emotion2.drop(columns=["stud_id"], errors="ignore")
emotion2_result = PCA.perform_pca_analysis(emotion2_clean, 0.3)

emotion2_result["merged_table"].to_csv(
    str(OUT_DIR / "emotion2_pca.csv"),
    encoding="utf-8-sig"
)

selected_items = emotion2_result["merged_table"].index.tolist()
valid_index = emotion2_clean[selected_items].dropna().index

emotion2_score = emotion2_result["pca_score"].copy()
emotion2_score["stud_id"] = emotion2.loc[valid_index, "stud_id"].to_numpy()
emotion2_score = emotion2_score[["stud_id", "PC1 Score"]]

emotion2_score.to_stata(
    str(OUT_DIR / "Emotion2_PCA.dta"),
    write_index=False
)
# Compound Skills Wave 2
compound2_clean = compound2.drop(columns=["stud_id"], errors="ignore")
compound2_result = PCA.perform_pca_analysis(compound2_clean, 0.3)

compound2_result["merged_table"].to_csv(
    str(OUT_DIR / "compound2_pca.csv"),
    encoding="utf-8-sig"
)

selected_items = compound2_result["merged_table"].index.tolist()
valid_index = compound2_clean[selected_items].dropna().index

compound2_score = compound2_result["pca_score"].copy()
compound2_score["stud_id"] = compound2.loc[valid_index, "stud_id"].to_numpy()
compound2_score = compound2_score[["stud_id", "PC1 Score"]]

compound2_score.to_stata(
    str(OUT_DIR / "Compound2_PCA.dta"),
    write_index=False
)

# Engage  Wave 2
engage2_clean = engage2.drop(columns=["stud_id"], errors="ignore")
engage2_result = PCA.perform_pca_analysis(engage2_clean, 0.25)

engage2_result["merged_table"].to_csv(
    str(OUT_DIR / "engage2_pca.csv"),
    encoding="utf-8-sig"
)

selected_items = engage2_result["merged_table"].index.tolist()
valid_index = engage2_clean[selected_items].dropna().index

engage2_score = engage2_result["pca_score"].copy()
engage2_score["stud_id"] = engage2.loc[valid_index, "stud_id"].to_numpy()
engage2_score = engage2_score[["stud_id", "PC1 Score"]]

engage2_score.to_stata(
    str(OUT_DIR / "Engage2_PCA.dta"),
    write_index=False
)